# Audit quality faults and build observable projections

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


In [ ]:
!pip install pyiceberg[s3fs]
!pip install prometheus-client requests

## Optional lakehouse and observability setup
The first cell installs the clients. Start the local catalog and metrics stack only for service-backed verification; operational projections and SLO calculations remain offline.

```bash
docker compose --profile lakehouse --profile observability up -d minio minio-init iceberg-rest prometheus grafana
```

Stop with `docker compose --profile lakehouse --profile observability down`; unavailable services use the local fallback.


**Set up a deterministic source run**


In [ ]:
import json
from pathlib import Path

import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
from fraudtwin.lakehouse import build_bronze_records, silver_rows

bronze = build_bronze_records(data.entities, data.behavior, data.manifest, include_oracle=False)
print({"bronze": len(bronze), "subjects": sorted({r.subject for r in bronze})})

**Run the core operation**


In [ ]:
display(pl.DataFrame([r.as_row() for r in bronze[:10]]))

**Measure and interpret the result**


In [ ]:
silver = silver_rows(bronze)
print({"silver": len(silver), "duplicates_removed": len(bronze) - len(silver)})

**Exercise a parameter or failure mode**


In [ ]:
quality = {
    "null_payloads": sum(not row["payload_json"] for row in silver),
    "unique_ids": len({row["record_id"] for row in silver}),
}
print(quality)

**Write a compact artifact and fingerprint**


In [ ]:
assert quality["null_payloads"] == 0
print("observable projection verified; oracle tables remain excluded")

**Verify invariants and clean up**


In [ ]:
# A compact inspection is more useful than printing an entire run.
print(
    payments.select(
        [
            c
            for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
            if c in payments.columns
        ]
    ).head(8)
)
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})

**Optional service integration**


In [ ]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

**Review the expected outcome**


In [ ]:
import os

try:
    from pyiceberg.catalog import load_catalog

    catalog = load_catalog(
        "fraudtwin",
        type="rest",
        uri=os.getenv("FRAUDTWIN_ICEBERG_CATALOG_URI", "http://localhost:8181"),
    )
    import requests

    namespaces = catalog.list_namespaces()
    prometheus = requests.get("http://localhost:9090/-/ready", timeout=3)
    print({"connected": True, "namespaces": namespaces, "prometheus": prometheus.status_code})
except Exception as exc:
    print({"connected": False, "offline_fallback": True, "reason": type(exc).__name__})

## Record the generated shape and tutorial contract.


In [ ]:
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 13,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

In [ ]:
assert len(bronze) > 0
print({"offline_fallback": True, "bronze_records": len(bronze)})